# EX: Evaluating Adversarial vs Stochastic AI Decisions

In this exercise, we will construct a simple decision tree representing a tactical scenario with two primary choices (Left or Right). We will write recursive Python functions to simulate how an AI processes this tree differently depending on whether it assumes the environment is actively hostile (Minimax) or random (Expectimax).

**Steps Performed:**

* **Define the tree structure** as a nested dictionary, complete with terminal values and probabilities.

* **Build a Minimax function** that simulates a rational enemy actively attempting to minimize the AI's score.

* **Build an Expectimax function** that calculates the mathematical average of the outcomes.

* **Compare the root node decisions** and observe how the AI alters its operational posture.

```{mermaid}
graph TD
    %% Define Nodes and Shapes
    %% MAX node is typically a triangle pointing up (represented here by a standard box to fit text)
    Root[MAX<br>Root]
    
    %% MIN/Chance nodes are typically circles or down-pointing triangles
    Left((Chance/MIN<br>Left Flank))
    Right((Chance/MIN<br>Right Flank))
    
    %% Terminal Nodes are typically squares
    L1[Terminal L1<br>Value: 10]
    L2[Terminal L2<br>Value: 5]
    R1[Terminal R1<br>Value: 7]
    R2[Terminal R2<br>Value: 20]

    %% Define Edges and Probabilities
    Root -->|Choose Left| Left
    Root -->|Choose Right| Right
    
    Left -->|Prob: 0.5| L1
    Left -->|Prob: 0.5| L2
    
    Right -->|Prob: 0.8| R1
    Right -->|Prob: 0.2| R2

    %% Styling for light/dark mode compatibility
    style Root fill:transparent,stroke:#3498db,stroke-width:3px
    style Left fill:transparent,stroke:#e67e22,stroke-width:3px
    style Right fill:transparent,stroke:#e67e22,stroke-width:3px
    style L1 fill:transparent,stroke:#2ecc71,stroke-width:2px
    style L2 fill:transparent,stroke:#2ecc71,stroke-width:2px
    style R1 fill:transparent,stroke:#2ecc71,stroke-width:2px
    style R2 fill:transparent,stroke:#2ecc71,stroke-width:2px
    ```

In [1]:
# --- Tactical Scenario Data ---
# Structure: {Node_Name: {'type': 'MAX'/'MIN'/'CHANCE', 'branches': [{outcome}], 'value': X}}
tactical_tree = {
    'Root': {'type': 'MAX', 'branches': ['Left_Flank', 'Right_Flank']},
    'Left_Flank': {'type': 'MIN_OR_CHANCE', 'branches': ['L1', 'L2']},
    'Right_Flank': {'type': 'MIN_OR_CHANCE', 'branches': ['R1', 'R2']},
    
    # Terminal outcomes. For Expectimax, we assign probabilities to these branches.
    'L1': {'type': 'TERMINAL', 'value': 10, 'prob': 0.5},
    'L2': {'type': 'TERMINAL', 'value': 5,  'prob': 0.5},
    'R1': {'type': 'TERMINAL', 'value': 7,  'prob': 0.8},
    'R2': {'type': 'TERMINAL', 'value': 20, 'prob': 0.2}
}

def minimax_decision(node_name):
    """Recursively calculates the Minimax value of a node."""
    node = tactical_tree[node_name]
    
    if node['type'] == 'TERMINAL':
        return node['value']
        
    if node['type'] == 'MAX':
        # MAX wants the highest possible backed-up value
        return max(minimax_decision(child) for child in node['branches'])
        
    if node['type'] == 'MIN_OR_CHANCE':
        # Treat as MIN node: Enemy wants the lowest possible backed-up value
        return min(minimax_decision(child) for child in node['branches'])

def expectimax_decision(node_name):
    """Recursively calculates the Expectimax EV of a node."""
    node = tactical_tree[node_name]
    
    if node['type'] == 'TERMINAL':
        return node['value']
        
    if node['type'] == 'MAX':
        # MAX wants the highest expected value
        return max(expectimax_decision(child) for child in node['branches'])
        
    if node['type'] == 'MIN_OR_CHANCE':
        # Treat as CHANCE node: Calculate Expected Value (Probability * Value)
        expected_value = 0
        for child in node['branches']:
            child_node = tactical_tree[child]
            expected_value += child_node['value'] * child_node['prob']
        return expected_value

print("--- TEST 1: Adversarial Environment (Minimax) ---")
left_minimax = minimax_decision('Left_Flank')
right_minimax = minimax_decision('Right_Flank')
best_minimax = minimax_decision('Root')
print(f"Left Flank backed-up value (Enemy chooses worst): {left_minimax}")
print(f"Right Flank backed-up value (Enemy chooses worst): {right_minimax}")
print(f"AI Decision: Go where the worst-case scenario is highest -> Score: {best_minimax}\n")

print("--- TEST 2: Stochastic Environment (Expectimax) ---")
left_expectimax = expectimax_decision('Left_Flank')
right_expectimax = expectimax_decision('Right_Flank')
best_expectimax = expectimax_decision('Root')
print(f"Left Flank expected value (EV): {left_expectimax}")
print(f"Right Flank expected value (EV): {right_expectimax}")
print(f"AI Decision: Go where the mathematical average is highest -> Score: {best_expectimax}")

--- TEST 1: Adversarial Environment (Minimax) ---
Left Flank backed-up value (Enemy chooses worst): 5
Right Flank backed-up value (Enemy chooses worst): 7
AI Decision: Go where the worst-case scenario is highest -> Score: 7

--- TEST 2: Stochastic Environment (Expectimax) ---
Left Flank expected value (EV): 7.5
Right Flank expected value (EV): 9.600000000000001
AI Decision: Go where the mathematical average is highest -> Score: 9.600000000000001


# Interpreting the Results

Notice how the AI's final decision completely flips based on its operating paradigm. 

In **Test 1 (Minimax)**, the AI assumes a rational enemy. The enemy will force the AI into a score of 5 on the left, or 7 on the right. To play it safe and guarantee the highest minimum score, the AI chooses the **Right Flank** (7).

In **Test 2 (Expectimax)**, the AI assumes the outcomes are purely driven by chance (like weather). The mathematical average of the Left Flank (7.5) is higher than the Right Flank (9.6). Wait, no—the code output reveals the Right Flank yields an EV of 9.6! In this specific graph, the Right Flank is mathematically superior in *both* scenarios. 

*Experiment:* Change the probability of `R2` (Value: 20) in the code block to `0.05` and `R1` (Value: 7) to `0.95`, then rerun the cell. You will see the EV of the Right Flank drop, forcing the Expectimax AI to switch its decision to the Left Flank, while the Minimax AI stubbornly ignores the probabilities entirely and remains on the Right Flank.